In [ ]:
import pandas as pd

df= pd.read_csv('2021_2022_MA_Region_Mobility_Report.csv')
df.head()

# Ce dataset contient des données de mobilité pour le Maroc (MA) sur la période 2021-2022.
# Les colonnes principales sont les suivantes :
# - `country_region_code` : Code du pays (ISO Alpha-2), ici "MA" pour le Maroc.
# - `country_region` : Nom du pays, ici "Morocco".
# - `place_id` : Identifiant unique pour une région ou un lieu spécifique.
# - `date` : Date des observations.
# - `retail_and_recreation_percent_change_from_baseline` : Variation en pourcentage de la mobilité dans les lieux de commerce et de loisirs par rapport à une base de référence.
# - `grocery_and_pharmacy_percent_change_from_baseline` : Variation en pourcentage de la mobilité dans les épiceries et pharmacies.
# - `parks_percent_change_from_baseline` : Variation en pourcentage de la mobilité dans les parcs.
# - `transit_stations_percent_change_from_baseline` : Variation en pourcentage de la mobilité dans les stations de transport en commun.
# - `workplaces_percent_change_from_baseline` : Variation en pourcentage de la mobilité sur les lieux de travail.
# - `residential_percent_change_from_baseline` : Variation en pourcentage de la mobilité dans les zones résidentielles.


# Les variations sont exprimées en pourcentage par rapport à une période de référence définie par Google.


,country_region_code,country_region,place_id,date,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
0,MA,Morocco,ChIJjcVRlmGICw0Rw_8sxIGT09k,2021-01-01,-33,21,-35,-26,-46,21
1,MA,Morocco,ChIJjcVRlmGICw0Rw_8sxIGT09k,2021-01-02,-26,27,-25,-14,-9,14
2,MA,Morocco,ChIJjcVRlmGICw0Rw_8sxIGT09k,2021-01-03,-22,26,-17,-10,-7,11
3,MA,Morocco,ChIJjcVRlmGICw0Rw_8sxIGT09k,2021-01-04,-22,34,-21,-6,-8,11
4,MA,Morocco,ChIJjcVRlmGICw0Rw_8sxIGT09k,2021-01-05,-28,23,-32,-14,-9,13


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2612 entries, 0 to 2611
Data columns (total 10 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   country_region_code                                 2612 non-null   object
 1   country_region                                      2612 non-null   object
 2   place_id                                            2612 non-null   object
 3   date                                                2612 non-null   object
 4   retail_and_recreation_percent_change_from_baseline  2612 non-null   int64 
 5   grocery_and_pharmacy_percent_change_from_baseline   2612 non-null   int64 
 6   parks_percent_change_from_baseline                  2612 non-null   int64 
 7   transit_stations_percent_change_from_baseline       2612 non-null   int64 
 8   workplaces_percent_change_from_baseline             2612 non-null   int64 
 9   resident

In [4]:
df.isna().sum()

country_region_code                                   0
country_region                                        0
place_id                                              0
date                                                  0
retail_and_recreation_percent_change_from_baseline    0
grocery_and_pharmacy_percent_change_from_baseline     0
parks_percent_change_from_baseline                    0
transit_stations_percent_change_from_baseline         0
workplaces_percent_change_from_baseline               0
residential_percent_change_from_baseline              0
dtype: int64

### csv_to_parquet.py 

Ce script a pour objectif de convertir un fichier CSV contenant des données de mobilité en un format Parquet optimisé pour le stockage et l'analyse. Voici les principales étapes réalisées par ce script :

1. **Initialisation de Spark** : Création d'une session Spark configurée pour optimiser les performances (compression Snappy, partitions, mémoire off-heap).

2. **Définition du schéma** : Spécification des types de données pour chaque colonne du fichier CSV afin de garantir une lecture cohérente.

3. **Lecture des données** : Chargement du fichier CSV en utilisant le schéma défini.

4. **Nettoyage des données** :
    - Identification et suppression des colonnes contenant plus de 50% de valeurs nulles.
    - Ajout de colonnes pour partitionner les données par année, mois et jour.

5. **Optimisation** :
    - Tri des données par colonnes de partitionnement.
    - Mise en cache des données pour améliorer les performances.

6. **Écriture en Parquet** : Sauvegarde des données nettoyées et optimisées au format Parquet, avec partitionnement par année, mois et jour.

7. **Statistiques** : Calcul et affichage des statistiques finales, comme le nombre total d'enregistrements et de colonnes.

Ce script est conçu pour traiter efficacement de grandes quantités de données, en les préparant pour une analyse rapide et scalable.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, dayofmonth, to_date, count
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, DoubleType
import sys
import logging

# Configuration du logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def create_spark_session():
    """Initialise et retourne une session Spark"""
    return SparkSession.builder \
        .appName("CSV_to_Parquet") \
        .master("local[*]") \
        .config("spark.sql.parquet.compression.codec", "snappy") \
        .config("spark.sql.shuffle.partitions", "4") \
        .config("spark.memory.offHeap.enabled", "true") \
        .config("spark.memory.offHeap.size", "2g") \
        .getOrCreate()

def define_schema():
    """Définit le schéma des données"""
    return StructType([
        StructField("country_region_code", StringType(), True),
        StructField("country_region", StringType(), True),
        StructField("place_id", StringType(), True),
        StructField("date", DateType(), True),
        StructField("retail_and_recreation_percent_change_from_baseline", IntegerType(), True),
        StructField("grocery_and_pharmacy_percent_change_from_baseline", IntegerType(), True),
        StructField("parks_percent_change_from_baseline", IntegerType(), True),
        StructField("transit_stations_percent_change_from_baseline", IntegerType(), True),
        StructField("workplaces_percent_change_from_baseline", IntegerType(), True),
        StructField("residential_percent_change_from_baseline", IntegerType(), True)
    ])

def process_data(spark, input_path, output_path):
    """Traite les données et les sauvegarde en format Parquet"""
    try:
        # Lecture du CSV avec schéma prédéfini
        logger.info("Lecture du fichier CSV...")
        df = spark.read.csv(input_path, header=True, schema=define_schema())

        # Identification et suppression des colonnes avec plus de 50% de valeurs NULL
        logger.info("Analyse des colonnes nulles...")
        total_rows = df.count()
        null_counts = df.select([count(col(c)).alias(c) for c in df.columns]).collect()[0]
        cols_to_drop = [c for c in df.columns if null_counts[c] < total_rows * 0.5]

        if cols_to_drop:
            logger.warning(f"Colonnes supprimées (>50% NULL): {cols_to_drop}")
            df = df.drop(*cols_to_drop)

        # Ajout des colonnes de partitionnement
        logger.info("Ajout des colonnes de partitionnement...")
        df = df.withColumn("année", year(col("date"))) \
               .withColumn("mois", month(col("date"))) \
               .withColumn("jour", dayofmonth(col("date")))

        # Tri des données par année, mois et jour
        df = df.orderBy("année", "mois", "jour")

        # Optimisation et cache
        df = df.repartition("année", "mois")
        df.cache()

        # Écriture en Parquet
        logger.info("Écriture des données en format Parquet...")
        df.write.mode("overwrite") \
            .partitionBy("année", "mois", "jour") \
            .parquet(output_path)

        logger.info(f"Données stockées avec succès dans {output_path}")

        # Statistiques sur les données
        logger.info("Statistiques finales:")
        logger.info(f"Nombre total d'enregistrements: {df.count()}")
        logger.info(f"Nombre de colonnes: {len(df.columns)}")

    except Exception as e:
        logger.error(f"Erreur lors du traitement: {str(e)}")
        raise

def main():
    """Fonction principale"""
    input_path = "/data/mobility/2021_2022_MA_Region_Mobility_Report.csv"
    output_path = "/data/mobility/mobility_parquet"

    spark = None
    try:
        spark = create_spark_session()
        process_data(spark, input_path, output_path)
    except Exception as e:
        logger.error(f"Erreur critique: {str(e)}")
        sys.exit(1)
    finally:
        if spark:
            spark.stop()
            logger.info("Session Spark fermée")

if __name__ == "__main__":
    main()

### producer.py

Ce script a pour objectif de lire des fichiers Parquet contenant des données de mobilité depuis un système de fichiers HDFS, de les traiter et de les envoyer sous forme de messages à un cluster Kafka. Voici les principales fonctionnalités de ce script :

1. **Connexion à HDFS** : Utilisation de `hdfs` pour accéder aux fichiers Parquet stockés dans un répertoire HDFS.

2. **Lecture des fichiers Parquet** : Parcours récursif d'un répertoire HDFS pour identifier et lire les fichiers Parquet.

3. **Traitement des données** :
    - Chargement des données dans un DataFrame pandas.
    - Conversion des colonnes de date et nettoyage des données.
    - Construction de messages JSON structurés pour chaque ligne de données.

4. **Connexion à Kafka** : Création d'un producteur Kafka pour envoyer les messages.

5. **Envoi des messages** :
    - Envoi des messages générés à un topic Kafka spécifique.
    - Gestion des erreurs et des tentatives de réenvoi en cas d'échec.

6. **Journalisation** : Utilisation de `logging` pour suivre les étapes du traitement, les erreurs et les statistiques sur les messages envoyés.

Ce script est conçu pour intégrer des données de mobilité dans un pipeline de traitement en temps réel, en les publiant sur Kafka pour une consommation ultérieure par d'autres services ou applications.

In [ ]:
import json
import time
import logging
import io
from datetime import datetime
import os

import pandas as pd
from kafka import KafkaProducer
from hdfs import InsecureClient

# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Configuration Kafka et topics
KAFKA_BOOTSTRAP_SERVERS = 'hadoop-master:9092'
TOPICS = {
    'mobility': 'mobility-data'
}

def create_producer():
    """Crée et retourne une instance de KafkaProducer"""
    try:
        producer = KafkaProducer(
            bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
            value_serializer=lambda v: json.dumps(v).encode('utf-8'),
            retries=5,
            acks='all',
            linger_ms=10,
            retry_backoff_ms=500
        )
        return producer
    except Exception as e:
        logger.error(f"Erreur création producteur Kafka: {str(e)}")
        raise

def list_parquet_files(client, directory):
    """
    Liste récursivement tous les fichiers Parquet dans le répertoire donné sur HDFS.
    """
    parquet_files = []
    try:
        items = client.list(directory, status=True)
        for item, status in items:
            path = os.path.join(directory, item)
            if status['type'] == 'FILE' and item.endswith('.parquet'):
                parquet_files.append(path)
            elif status['type'] == 'DIRECTORY':
                # Parcours récursif du sous-dossier
                parquet_files.extend(list_parquet_files(client, path))
        return parquet_files
    except Exception as e:
        logger.error(f"Erreur lors du listing des fichiers dans {directory}: {str(e)}")
        raise

def process_mobility_data(hdfs_dir):
    """
    Traite les données de mobilité depuis tous les fichiers Parquet du répertoire HDFS.
    Pour chaque fichier trouvé, il lit le contenu avec pandas et génère un message par ligne.
    """
    try:
        # Configuration du client HDFS
        client = InsecureClient('http://hadoop-master:9870', user='user')
        logger.info(f"Listing des fichiers Parquet dans {hdfs_dir}")
        parquet_files = list_parquet_files(client, hdfs_dir)
        if not parquet_files:
            raise FileNotFoundError(f"Aucun fichier Parquet trouvé dans : {hdfs_dir}")
        else:
            logger.info(f"{len(parquet_files)} fichier(s) Parquet trouvé(s)")

        # Pour chaque fichier Parquet trouvé
        for file_path in parquet_files:
            logger.info(f"Lecture du fichier : {file_path}")
            with client.read(file_path) as reader:
                # Charger le contenu dans un DataFrame pandas
                df = pd.read_parquet(io.BytesIO(reader.read()))

            # Conversion de la colonne date
            if 'date' in df.columns:
                df['date'] = pd.to_datetime(df['date'])

            # Pour chaque ligne, construire le message à envoyer
            for _, row in df.iterrows():
                try:
                    message = {
                        "type": "mobility",
                        "timestamp": datetime.now().isoformat(),
                        "data": {
                            "date": row["date"].isoformat() if 'date' in row and pd.notnull(row["date"]) else None,
                            "country": str(row["country_region"]) if "country_region" in row else None,
                            "metrics": {
                                "retail": float(row["retail_and_recreation_percent_change_from_baseline"]) if "retail_and_recreation_percent_change_from_baseline" in row else None,
                                "grocery": float(row["grocery_and_pharmacy_percent_change_from_baseline"]) if "grocery_and_pharmacy_percent_change_from_baseline" in row else None,
                                "parks": float(row["parks_percent_change_from_baseline"]) if "parks_percent_change_from_baseline" in row else None,
                                "transit": float(row["transit_stations_percent_change_from_baseline"]) if "transit_stations_percent_change_from_baseline" in row else None,
                                "workplaces": float(row["workplaces_percent_change_from_baseline"]) if "workplaces_percent_change_from_baseline" in row else None,
                                "residential": float(row["residential_percent_change_from_baseline"]) if "residential_percent_change_from_baseline" in row else None
                            }
                        }
                    }
                    yield message
                except (ValueError, TypeError) as e:
                    logger.warning(f"Skipping row due to data error: {e}")
                    continue
    except Exception as e:
        logger.error(f"Erreur traitement données mobilité: {str(e)}")
        raise

def send_message(producer, topic, data):
    """Envoie un message à Kafka"""
    try:
        future = producer.send(topic, value=data)
        result = future.get(timeout=10)
        logger.debug(f"Message sent to partition {result.partition}, offset {result.offset}")
        return True
    except Exception as e:
        logger.error(f"Erreur envoi message Kafka: {str(e)}")
        return False

def main():
    producer = None
    try:
        producer = create_producer()
        logger.info("Producteur Kafka créé avec succès")

        # Chemin du répertoire contenant les fichiers Parquet sur HDFS
        hdfs_dir = "/data/mobility/mobility_parquet"
        logger.info(f"Début du traitement des données Parquet depuis: {hdfs_dir}")

        mobility_data = process_mobility_data(hdfs_dir)
        messages_sent = 0

        for data in mobility_data:
            success = send_message(producer, TOPICS['mobility'], data)
            if success:
                messages_sent += 1
                if messages_sent % 100 == 0:
                    logger.info(f"Nombre de messages envoyés: {messages_sent}")
            time.sleep(0.5)  # Limitation de débit

        logger.info(f"Traitement terminé. Total messages envoyés: {messages_sent}")

    except Exception as e:
        logger.error(f"Erreur critique: {str(e)}")
    finally:
        if producer:
            try:
                producer.flush()
                producer.close(timeout=5)
                logger.info("Producteur Kafka fermé")
            except Exception as e:
                logger.error(f"Erreur lors de la fermeture du producteur: {str(e)}")

if __name__ == "__main__":
    main()

### train_model.py

Ce script a pour objectif d'entraîner un modèle de machine learning à partir des données de mobilité stockées au format Parquet dans HDFS. Voici les principales étapes réalisées par ce script :

1. **Chargement des données** :
    - Connexion à HDFS pour lister et charger les fichiers Parquet contenant les données de mobilité.
    - Préparation des données en supprimant les valeurs manquantes et en sélectionnant les colonnes pertinentes.

2. **Préparation des caractéristiques** :
    - Utilisation de `VectorAssembler` pour combiner les colonnes sélectionnées en un vecteur de caractéristiques (`features`).

3. **Entraînement du modèle** :
    - Entraînement d'un modèle de régression linéaire (`LinearRegression`) pour prédire les variations de mobilité à partir des données disponibles.

4. **Évaluation du modèle** :
    - Calcul des métriques de performance telles que RMSE (Root Mean Squared Error), R² (Coefficient de détermination) et MAE (Mean Absolute Error).
    - Affichage des résultats pour évaluer la qualité du modèle.

5. **Sauvegarde du modèle** :
    - Enregistrement du modèle entraîné dans HDFS pour une utilisation ultérieure dans des pipelines de prédiction.

Ce script est conçu pour automatiser le processus d'entraînement et d'évaluation des modèles, en s'appuyant sur Spark pour traiter efficacement de grandes quantités de données.

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from hdfs import InsecureClient
import os
import logging
from pyspark.sql import SparkSession  # Import SparkSession

# Configuration du logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def list_parquet_files(client, directory):
    """
    Liste récursivement tous les fichiers Parquet dans le répertoire donné sur HDFS.
    """
    parquet_files = []
    try:
        items = client.list(directory, status=True)
        for item, status in items:
            path = os.path.join(directory, item)
            if status['type'] == 'FILE' and item.endswith('.parquet'):
                parquet_files.append(f"hdfs://hadoop-master:9000{path}")
            elif status['type'] == 'DIRECTORY':
                # Parcours récursif du sous-dossier
                parquet_files.extend(list_parquet_files(client, path))
        return parquet_files
    except Exception as e:
        logger.error(f"Erreur lors du listing des fichiers dans {directory}: {str(e)}")
        return []

def load_and_prepare_data(hdfs_dir, spark):  # Added spark parameter
    """
    Charge et prépare les données de mobilité depuis tous les fichiers Parquet dans HDFS.
    """
    try:
        # Configuration du client HDFS
        client = InsecureClient('http://hadoop-master:9870', user='root')  # Utilisateur root
        logger.info(f"Listing des fichiers Parquet dans {hdfs_dir}")
        parquet_files = list_parquet_files(client, hdfs_dir)
        if not parquet_files:
            logger.error(f"Aucun fichier Parquet trouvé dans : {hdfs_dir}")
            return None
        else:
            logger.info(f"{len(parquet_files)} fichier(s) Parquet trouvé(s) : {parquet_files}")

        # Charger tous les fichiers Parquet avec Spark
        df = spark.read.parquet(*parquet_files).limit(7)  # Limiter à 10 lignes pour test
        df = df.dropna(thresh=5)
        feature_cols = [c for c in df.columns
                       if c not in ["country_region_code", "place_id", "date",
                                   "country_region", "retail_and_recreation_percent_change_from_baseline"]]
        assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
        return assembler.transform(df)
    except Exception as e:
        logger.error(f"Erreur lors du chargement des données : {str(e)}")
        return None

def train_and_evaluate_model(model, data, model_name, model_path):
    logger.info(f"Entraînement du modèle {model_name}...")
    trained_model = model.fit(data)

    # Evaluation du modèle
    predictions = trained_model.transform(data)
    evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="retail_and_recreation_percent_change_from_baseline")

    rmse = evaluator.setMetricName("rmse").evaluate(predictions)
    r2 = evaluator.setMetricName("r2").evaluate(predictions)
    mae = evaluator.setMetricName("mae").evaluate(predictions)

    # Affichage des métriques dans la console
    logger.info(f"Metrics pour le modèle {model_name}:")
    logger.info(f"  RMSE (Root Mean Squared Error) : {rmse}")
    logger.info(f"  R² (Coefficient de détermination) : {r2}")
    logger.info(f"  MAE (Mean Absolute Error) : {mae}")

    model_save_path = f"hdfs://hadoop-master:9000{model_path}/{model_name}"
    trained_model.save(model_save_path)
    logger.info(f"Modèle {model_name} sauvegardé dans {model_save_path}")

def main():
    # Créer une SparkSession avec ressources minimales
    spark = SparkSession.builder \
        .appName("MobilityModelTraining") \
        .master("local[*]") \
        .config("spark.driver.memory", "4g") \
        .config("spark.executor.memory", "4g") \
        .config("spark.executor.cores", "4") \
        .config("spark.sql.shuffle.partitions", "4") \
        .getOrCreate()

    config = {
        "input_path": "/data/mobility/mobility_parquet",  # Chemin HDFS sans préfixe
        "models_path": "/data/mobility/models"
    }

    try:
        # Charger et préparer les données
        prepared_data = load_and_prepare_data(config["input_path"], spark)  # Pass spark to the function
        if prepared_data is None:
            logger.error("Échec de la préparation des données. Arrêt.")
            return

        # Configurer et entraîner le modèle
        model = LinearRegression(featuresCol="features", labelCol="retail_and_recreation_percent_change_from_baseline")
        train_and_evaluate_model(model, prepared_data, "LinearRegression", config["models_path"])

        logger.info("Entraînement terminé avec succès")
    except Exception as e:
        logger.error(f"Erreur dans main : {str(e)}", exc_info=True)

if __name__ == "__main__":
    main()

### test3.py

Ce script a pour objectif de traiter des données de mobilité en temps réel à partir d'un flux Kafka, d'appliquer un modèle de machine learning pour effectuer des prédictions, et de sauvegarder les résultats dans une base de données PostgreSQL. Voici les principales étapes réalisées par ce script :

1. **Initialisation Spark** :
    - Création d'une session Spark configurée pour le traitement en streaming.
    - Vérification de la disponibilité du driver PostgreSQL et test de connexion à la base de données.

2. **Chargement du modèle** :
    - Récupération d'un modèle de régression linéaire préalablement entraîné et stocké dans HDFS.

3. **Traitement des données Kafka** :
    - Lecture des messages depuis un topic Kafka contenant des données de mobilité.
    - Parsing des messages JSON pour extraire les métriques pertinentes.
    - Transformation des données en vecteurs de caractéristiques pour les prédictions.

4. **Prédictions en temps réel** :
    - Application du modèle de machine learning sur les données traitées pour prédire la mobilité résidentielle.
    - Formatage des résultats sous forme de JSON.

5. **Sauvegarde dans PostgreSQL** :
    - Validation et écriture des prédictions dans une table staging de PostgreSQL.
    - Gestion des erreurs et journalisation des étapes.

6. **Gestion des flux** :
    - Configuration d'un flux Spark Streaming pour traiter les données en continu.
    - Utilisation d'un emplacement de checkpoint pour garantir la tolérance aux pannes.

Ce script est conçu pour intégrer des prédictions en temps réel dans un pipeline de données, en combinant Kafka, Spark et PostgreSQL pour une analyse rapide et scalable.

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from hdfs import InsecureClient
import os
import logging
from pyspark.sql import SparkSession  # Import SparkSession

# Configuration du logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def list_parquet_files(client, directory):
    """
    Liste récursivement tous les fichiers Parquet dans le répertoire donné sur HDFS.
    """
    parquet_files = []
    try:
        items = client.list(directory, status=True)
        for item, status in items:
            path = os.path.join(directory, item)
            if status['type'] == 'FILE' and item.endswith('.parquet'):
                parquet_files.append(f"hdfs://hadoop-master:9000{path}")
            elif status['type'] == 'DIRECTORY':
                # Parcours récursif du sous-dossier
                parquet_files.extend(list_parquet_files(client, path))
        return parquet_files
    except Exception as e:
        logger.error(f"Erreur lors du listing des fichiers dans {directory}: {str(e)}")
        return []

def load_and_prepare_data(hdfs_dir, spark):  # Added spark parameter
    """
    Charge et prépare les données de mobilité depuis tous les fichiers Parquet dans HDFS.
    """
    try:
        # Configuration du client HDFS
        client = InsecureClient('http://hadoop-master:9870', user='root')  # Utilisateur root
        logger.info(f"Listing des fichiers Parquet dans {hdfs_dir}")
        parquet_files = list_parquet_files(client, hdfs_dir)
        if not parquet_files:
            logger.error(f"Aucun fichier Parquet trouvé dans : {hdfs_dir}")
            return None
        else:
            logger.info(f"{len(parquet_files)} fichier(s) Parquet trouvé(s) : {parquet_files}")

        # Charger tous les fichiers Parquet avec Spark
        df = spark.read.parquet(*parquet_files).limit(7)  # Limiter à 10 lignes pour test
        df = df.dropna(thresh=5)
        feature_cols = [c for c in df.columns
                       if c not in ["country_region_code", "place_id", "date",
                                   "country_region", "retail_and_recreation_percent_change_from_baseline"]]
        assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
        return assembler.transform(df)
    except Exception as e:
        logger.error(f"Erreur lors du chargement des données : {str(e)}")
        return None

def train_and_evaluate_model(model, data, model_name, model_path):
    logger.info(f"Entraînement du modèle {model_name}...")
    trained_model = model.fit(data)

    # Evaluation du modèle
    predictions = trained_model.transform(data)
    evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="retail_and_recreation_percent_change_from_baseline")

    rmse = evaluator.setMetricName("rmse").evaluate(predictions)
    r2 = evaluator.setMetricName("r2").evaluate(predictions)
    mae = evaluator.setMetricName("mae").evaluate(predictions)

    # Affichage des métriques dans la console
    logger.info(f"Metrics pour le modèle {model_name}:")
    logger.info(f"  RMSE (Root Mean Squared Error) : {rmse}")
    logger.info(f"  R² (Coefficient de détermination) : {r2}")
    logger.info(f"  MAE (Mean Absolute Error) : {mae}")

    model_save_path = f"hdfs://hadoop-master:9000{model_path}/{model_name}"
    trained_model.save(model_save_path)
    logger.info(f"Modèle {model_name} sauvegardé dans {model_save_path}")

def main():
    # Créer une SparkSession avec ressources minimales
    spark = SparkSession.builder \
        .appName("MobilityModelTraining") \
        .master("local[*]") \
        .config("spark.driver.memory", "4g") \
        .config("spark.executor.memory", "4g") \
        .config("spark.executor.cores", "4") \
        .config("spark.sql.shuffle.partitions", "4") \
        .getOrCreate()

    config = {
        "input_path": "/data/mobility/mobility_parquet",  # Chemin HDFS sans préfixe
        "models_path": "/data/mobility/models"
    }

    try:
        # Charger et préparer les données
        prepared_data = load_and_prepare_data(config["input_path"], spark)  # Pass spark to the function
        if prepared_data is None:
            logger.error("Échec de la préparation des données. Arrêt.")
            return

        # Configurer et entraîner le modèle
        model = LinearRegression(featuresCol="features", labelCol="retail_and_recreation_percent_change_from_baseline")
        train_and_evaluate_model(model, prepared_data, "LinearRegression", config["models_path"])

        logger.info("Entraînement terminé avec succès")
    except Exception as e:
        logger.error(f"Erreur dans main : {str(e)}", exc_info=True)

if __name__ == "__main__":
    main()

root@hadoop-master:~# cat test3.py
#!/usr/bin/env python3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_json, lit, current_timestamp, struct, when
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, ArrayType
from pyspark.ml.regression import LinearRegressionModel
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import udf
from pyspark.ml.linalg import Vectors
import logging
from datetime import datetime
from hdfs import InsecureClient
import sys
import json

# Configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration PostgreSQL avec driver intégré
POSTGRES_CONFIG = {
    "url": "jdbc:postgresql://postgres:5432/bigdata",
    "user": "spark_user",
    "password": "SecurePass123!",
    "driver": "org.postgresql.Driver",
    "predictions_table": "predictions",
    "staging_table": "predictions_staging",
    "connection_properties": {
        "ssl": "false",
        "application_name": "spark_streaming"
    }
}

HDFS_PATHS = {
    "models_path": "hdfs:///data/mobility/models",
}

# UDF pour conversion Vector -> Array
def vector_to_array(v):
    try:
        return v.toArray().tolist()
    except Exception as e:
        logger.error(f"Vector conversion error: {str(e)}")
        return None

# Initialisation Spark avec configuration du driver
def create_spark_session():
    spark = SparkSession.builder \
        .appName("Mobility_RealTime_Prediction") \
        .master("local[2]") \
        .config("spark.jars", "/root/postgresql-42.7.3.jar") \
        .config("spark.driver.extraClassPath", "/root/postgresql-42.7.3.jar") \
        .config("spark.executor.extraClassPath", "/root/postgresql-42.7.3.jar") \
        .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1") \
        .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true") \
        .getOrCreate()

    spark.udf.register("vector_to_array", vector_to_array, ArrayType(DoubleType()))
    return spark

# Vérification du driver PostgreSQL
def verify_postgres_driver():
    try:
        from py4j.java_gateway import java_import
        gw = SparkSession.builder.getOrCreate().sparkContext._gateway
        java_import(gw.jvm, "org.postgresql.Driver")
        logger.info("PostgreSQL driver verified successfully")
        return True
    except Exception as e:
        logger.error(f"Driver verification failed: {str(e)}")
        return False

# Test de connexion PostgreSQL
def test_postgres_connection(spark):
    try:
        test_df = spark.read \
            .format("jdbc") \
            .option("url", POSTGRES_CONFIG["url"]) \
            .option("query", "SELECT 1 as test") \
            .option("user", POSTGRES_CONFIG["user"]) \
            .option("password", POSTGRES_CONFIG["password"]) \
            .option("driver", POSTGRES_CONFIG["driver"]) \
            .load()
        return test_df.first()["test"] == 1
    except Exception as e:
        logger.error(f"Connection test failed: {str(e)}")
        return False

# Chargement du modèle
def load_models(spark, hdfs_client):
    logger.info("Loading model from HDFS...")
    model_path = f"{HDFS_PATHS['models_path']}/LinearRegression"
    try:
        if hdfs_client.status(model_path.replace("hdfs://", "/"), strict=False):
            model = LinearRegressionModel.load(model_path)
            logger.info(f"Model loaded from {model_path}")
            return model
    except Exception as e:
        logger.error(f"Model loading failed: {str(e)}", exc_info=True)
        raise

# Sauvegarde des données avec gestion du driver
def save_to_postgres(df, epoch_id):
    try:
        # Validation des données
        df = df.withColumn("is_valid",
            when(col("features_json").isNotNull(), True).otherwise(False))

        # Écriture dans la table staging
        (df.write
            .format("jdbc")
            .option("url", POSTGRES_CONFIG["url"])
            .option("dbtable", POSTGRES_CONFIG["staging_table"])
            .option("user", POSTGRES_CONFIG["user"])
            .option("password", POSTGRES_CONFIG["password"])
            .option("driver", POSTGRES_CONFIG["driver"])
            .options(**POSTGRES_CONFIG["connection_properties"])
            .mode("append")
            .save())

        logger.info(f"Batch {epoch_id} saved to staging table")
    except Exception as e:
        logger.error(f"PostgreSQL save failed: {str(e)}", exc_info=True)
        raise

# Traitement des données
def process_batch(batch_df, batch_id, model, spark):
    epoch_id = int(datetime.now().timestamp())
    logger.info(f"Processing batch {batch_id} (epoch_id: {epoch_id})")

    try:
        # Schéma des données Kafka
        schema = StructType([
            StructField("type", StringType()),
            StructField("timestamp", StringType()),
            StructField("data", StructType([
                StructField("date", DateType()),
                StructField("country", StringType()),
                StructField("metrics", StructType([
                    StructField("retail", DoubleType()),
                    StructField("grocery", DoubleType()),
                    StructField("parks", DoubleType()),
                    StructField("transit", DoubleType()),
                    StructField("workplaces", DoubleType()),
                    StructField("residential", DoubleType())
                ]))
            ]))
        ])

        # Parsing des données
        parsed_df = batch_df.select(
            from_json(col("value").cast("string"), schema).alias("json")
        ).select("json.data.*")

        # Extraction des métriques
        metrics_df = parsed_df.select(
            col("date"),
            col("country"),
            col("metrics.retail").alias("retail"),
            col("metrics.grocery").alias("grocery"),
            col("metrics.parks").alias("parks"),
            col("metrics.transit").alias("transit"),
            col("metrics.workplaces").alias("workplaces"),
            col("metrics.residential").alias("residential")
        )

        # Feature engineering
        feature_cols = ["retail", "grocery", "parks", "transit", "workplaces", "residential"]
        assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
        processed_df = assembler.transform(metrics_df)

        # Conversion des features
        vector_to_array_udf = udf(vector_to_array, ArrayType(DoubleType()))
        processed_df = processed_df.withColumn("features_array", vector_to_array_udf(col("features")))

        # Formatage pour PostgreSQL
        predictions = processed_df.select(
            col("date"),
            col("country"),
            col("residential"),
            to_json(struct(
                lit("vector").alias("type"),
                lit(len(feature_cols)).alias("size"),
                col("features_array").alias("values")
            )).alias("features_json"),
            lit("LinearRegression").alias("model"),
            lit(epoch_id).alias("epoch_id"),
            current_timestamp().alias("created_at")
        )

        # Sauvegarde
        save_to_postgres(predictions, epoch_id)

    except Exception as e:
        logger.error(f"Batch processing failed: {str(e)}", exc_info=True)
        raise

def main():
    spark = None
    query = None
    hdfs_client = InsecureClient('http://hadoop-master:9870', user='root')

    try:
        # Initialisation
        spark = create_spark_session()

        if not verify_postgres_driver():
            logger.error("PostgreSQL driver not available")
            sys.exit(1)

        if not test_postgres_connection(spark):
            logger.error("PostgreSQL connection test failed")
            sys.exit(1)

        # Chargement modèle
        model = load_models(spark, hdfs_client)

        # Configuration du flux Kafka
        df_stream = spark.readStream \
            .format("kafka") \
            .option("kafka.bootstrap.servers", "hadoop-master:9092") \
            .option("subscribe", "mobility-data") \
            .option("startingOffsets", "latest") \
            .option("failOnDataLoss", "false") \
            .load()

        # Démarrage du traitement
        query = df_stream.writeStream \
            .foreachBatch(lambda df, id: process_batch(df, id, model, spark)) \
            .outputMode("update") \
            .option("checkpointLocation", "hdfs://hadoop-master:9000/checkpoints/mobility_prediction") \
            .start()

        logger.info("Stream processing started successfully")
        query.awaitTermination()

    except KeyboardInterrupt:
        logger.info("Received shutdown signal...")
    except Exception as e:
        logger.error(f"Fatal error: {str(e)}", exc_info=True)
    finally:
        try:
            if query and query.isActive:
                query.stop()
        except Exception as e:
            logger.error(f"Error stopping query: {str(e)}")

        try:
            if spark:
                spark.stop()
        except Exception as e:
            logger.error(f"Error stopping Spark: {str(e)}")

        logger.info("Application shutdown completed")

if __name__ == "__main__":
    main()

### app.py 

Ce script est une application Streamlit qui sert de tableau de bord interactif pour visualiser les prédictions issues d'un modèle de machine learning basé sur la mobilité. Voici les principales fonctionnalités de ce script :

1. **Configuration de la page** : Définition du titre, de la mise en page et de l'état initial de la barre latérale.

2. **Style personnalisé** : Ajout de CSS pour un fond animé et une interface utilisateur améliorée.

3. **Connexion à PostgreSQL** : Établissement d'une connexion sécurisée à une base de données PostgreSQL pour récupérer les données.

4. **Chargement des données** : Exécution d'une requête SQL pour extraire les prédictions et les transformer en DataFrame pandas.

5. **Visualisation des données** :
    - Affichage des indicateurs clés (KPIs) comme le nombre total de prédictions, les pays couverts et la dernière mise à jour.
    - Graphiques interactifs pour explorer les tendances de mobilité résidentielle par pays, période ou modèle.
    - Histogrammes et boîtes à moustaches pour analyser la distribution des scores de mobilité.

6. **Filtres interactifs** :
    - Sélection de pays, plages de dates et modèles spécifiques.
    - Ajustement d'un seuil de mobilité pour identifier les pays avec une mobilité élevée.

7. **Données brutes** : Option pour afficher un aperçu des données brutes.

Ce tableau de bord est conçu pour fournir une analyse approfondie et interactive des données de mobilité, facilitant ainsi la prise de décision basée sur les prédictions du modèle.

In [ ]:
import streamlit as st
import pandas as pd
import psycopg2
import plotly.express as px
from datetime import datetime

# Page Config
st.set_page_config(
    page_title="Pandémie & Mobilité Dashboard",
    layout="wide",
    initial_sidebar_state="expanded"
)

# === FOND ANIMÉ DYNAMIQUE (CSS) ===
st.markdown("""
    <style>
    body {
        background: linear-gradient(-45deg, #1f1c2c, #928dab, #0f2027, #203a43);
        background-size: 400% 400%;
        animation: gradient 15s ease infinite;
        color: white;
    }

    @keyframes gradient {
        0% {background-position: 0% 50%;}
        50% {background-position: 100% 50%;}
        100% {background-position: 0% 50%;}
    }

    .stApp {
        background: transparent;
    }

    .block-container {
        padding: 2rem;
        background-color: rgba(0, 0, 0, 0.55);
        border-radius: 15px;
    }

    h1, h2, h3, h4 {
        color: #00f2fe;
        text-shadow: 1px 1px 2px black;
    }
    </style>
""", unsafe_allow_html=True)

# Connexion PostgreSQL
def connect_to_postgres():
    try:
        conn = psycopg2.connect(
            host="postgres",
            port=5432,
            database="bigdata",
            user="spark_user",
            password="SecurePass123!"
        )
        return conn
    except Exception as e:
        st.error(f"Connexion PostgreSQL échouée : {e}")
        st.stop()

# Chargement des données
def fetch_data(conn):
    try:
        query = "SELECT * FROM predictions;"
        return pd.read_sql(query, conn)
    except Exception as e:
        st.error(f"Erreur lors de la récupération des données : {e}")
        return pd.DataFrame()

# Connexion et chargement
conn = connect_to_postgres()
df = fetch_data(conn)
conn.close()

if df.empty:
    st.warning("Aucune donnée à afficher.")
else:
    df["date"] = pd.to_datetime(df["date"])
    df["created_at"] = pd.to_datetime(df["created_at"])

    st.title("Dashboard Pandémie & Mobilité")
    st.markdown("## Visualisation des prédictions issues du modèle ML basé sur la mobilité")

    # KPIs rapides
    total_rows = len(df)
    last_update = df["created_at"].max()
    countries = df["country"].nunique()

    col1, col2, col3 = st.columns(3)
    col1.metric("Total Prédictions", f"{total_rows}")
    col2.metric("Pays couverts", f"{countries}")
    col3.metric("Dernière mise à jour", f"{last_update.strftime('%Y-%m-%d %H:%M')}")

    st.markdown("---")

    # Sélection de pays
    selected_country = st.sidebar.selectbox("Pays à explorer", sorted(df["country"].unique()))
    df_country = df[df["country"] == selected_country]

    st.subheader(f"Mobilité résidentielle - {selected_country}")
    fig = px.line(df_country, x="date", y="residential", title=f"Évolution de la mobilité résidentielle ({selected_country})", color_discrete_sequence=["#00f2fe"])
    st.plotly_chart(fig, use_container_width=True)

    # Sélection de dates
    st.sidebar.markdown("Filtrage temporel")
    min_date = df_country["date"].min().date()
    max_date = df_country["date"].max().date()
    date_range = st.sidebar.date_input("Sélectionnez une plage de dates", (min_date, max_date), min_value=min_date, max_value=max_date)

    if len(date_range) == 2:
        start_date, end_date = date_range
        df_filtered = df_country[(df_country["date"] >= pd.to_datetime(start_date)) & (df_country["date"] <= pd.to_datetime(end_date))]

        st.subheader(f"Détail de la période {start_date} ➡️ {end_date}")
        fig = px.area(df_filtered, x="date", y="residential", color="model", title="Comparaison des prédictions", color_discrete_sequence=px.colors.qualitative.Dark24)
        st.plotly_chart(fig, use_container_width=True)

    # Filtrage par modèle
    st.sidebar.markdown("Modèle de prédiction")
    selected_model = st.sidebar.selectbox("Choisissez le modèle", df["model"].unique())
    df_model = df[df["model"] == selected_model]

    st.subheader(f"Distribution - {selected_model}")
    fig = px.histogram(df_model, x="residential", nbins=30, title="Distribution des scores de mobilité résidentielle", color_discrete_sequence=["#f77062"])
    st.plotly_chart(fig, use_container_width=True)

    # Seuil interactif
    st.sidebar.markdown("Seuil de mobilité")
    threshold = st.sidebar.slider("Valeur minimale", float(df["residential"].min()), float(df["residential"].max()), step=0.5)
    high_mobility = df[df["residential"] >= threshold]

    st.subheader(f"Pays avec mobilité résidentielle > {threshold}")
    fig = px.box(high_mobility, x="country", y="residential", color="model", points="all", color_discrete_sequence=px.colors.sequential.Magma)
    st.plotly_chart(fig, use_container_width=True)

    # Données brutes en option
    with st.expander("Données brutes (preview)"):
        st.dataframe(df_model.head(50))